In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.config import (
    CAPTION_DROPOUT,
    DATASET_DIR,
    FINETRAINERS_REPO,
    FINETRAINERS_TAG,
    LR_SCHEDULER,
    MAX_GRAD_NORM,
    MIN_FRAMES,
    MODEL_ID,
    OUTPUT_DIR,
    SEED,
    TARGET_HEIGHT,
    TARGET_WIDTH,
    TRIGGER_TOKEN,
)

HF_TOKEN = os.getenv("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
else:
    print("WARNING: HF_TOKEN not set — gated model downloads may fail")

print("Configuration")
print("=" * 50)
print(f"Model        : {MODEL_ID}")
print(f"Dataset      : {DATASET_DIR}")
print(f"Output       : {OUTPUT_DIR}")
print(f"Trigger token: {TRIGGER_TOKEN}")

/home/ubuntu/MultiModal_Video_Model/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Configuration
Model        : hunyuanvideo-community/HunyuanVideo
Dataset      : /home/ubuntu/MultiModal_Video_Model/data/processed
Output       : /home/ubuntu/MultiModal_Video_Model/output/lora_weights
Trigger token: ohwx


In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA is required — no GPU detected!"

from src.training.gpu_utils import get_supported_dtype, is_fp8_supported
from src.training.train import get_resolution_and_fp8

num_gpus = torch.cuda.device_count()
total_vram = 0
sm_major = torch.cuda.get_device_properties(0).major

print(f"{'GPU':>5}  {'Name':<30}  {'VRAM':>8}  {'SM':>5}")
print("-" * 56)
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / 1e9
    total_vram += vram_gb
    print(f"{i:>5}  {props.name:<30}  {vram_gb:>7.1f}G  {props.major}.{props.minor:>3}")

per_gpu_mem = total_vram / num_gpus
mixed_precision = get_supported_dtype()
needs_sharding = num_gpus > 1 and per_gpu_mem < 35

resolution_buckets, use_fp8 = get_resolution_and_fp8(
    per_gpu_mem, fp8_supported=is_fp8_supported()
)

print(f"\n{'='*56}")
print(f"GPUs: {num_gpus}  |  Per-GPU: {per_gpu_mem:.0f} GB  |  Total: {total_vram:.0f} GB")
print(f"Mixed precision: {mixed_precision}  |  FP8: {use_fp8}")
print(f"Resolution buckets: {resolution_buckets}")
print(f"Needs DeepSpeed sharding: {needs_sharding}")
print(f"{'='*56}")

In [ ]:
from src.training.train import setup_finetrainers

finetrainers_dir = str(PROJECT_ROOT / "finetrainers")

train_script = setup_finetrainers(
    install_dir=finetrainers_dir,
    repo_url=FINETRAINERS_REPO,
    tag=FINETRAINERS_TAG,
)

print(f"\n✅ train.py: {train_script}")

In [ ]:
from src.training.validate import validate_dataset

videos_txt = DATASET_DIR / "videos.txt"
prompts_txt = DATASET_DIR / "prompts.txt"

assert videos_txt.exists(), f"videos.txt not found at {videos_txt} — run notebook 02 first!"
assert prompts_txt.exists(), f"prompts.txt not found at {prompts_txt} — run notebook 02 first!"

with open(videos_txt) as f:
    video_lines = f.read().strip().split("\n")
with open(prompts_txt) as f:
    prompt_lines = f.read().strip().split("\n")

print(f"Videos  : {len(video_lines)}")
print(f"Captions: {len(prompt_lines)}")
print(f"\nSample captions:")
for i, cap in enumerate(prompt_lines[:3]):
    print(f"  [{i}] {cap[:120]}...")

print()
is_valid = validate_dataset(
    str(DATASET_DIR), target_w=TARGET_WIDTH, target_h=TARGET_HEIGHT, min_frames=MIN_FRAMES
)
assert is_valid, "Dataset validation failed — fix errors above before training!"

In [ ]:
# ✏️ TRAINING HYPERPARAMETERS — edit these before launching
TRAIN_STEPS = 1500
LORA_RANK = 64
LORA_ALPHA = 64
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1       # reduced: multi-GPU effective batch = num_gpus * batch_size
WARMUP_STEPS = 100
CHECKPOINTING_STEPS = 500

effective_batch = num_gpus * BATCH_SIZE * GRAD_ACCUM_STEPS

print("Training Settings")
print("=" * 50)
print(f"Steps           : {TRAIN_STEPS}")
print(f"LoRA rank/alpha : {LORA_RANK}/{LORA_ALPHA}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Batch size      : {BATCH_SIZE} per GPU x {num_gpus} GPUs x {GRAD_ACCUM_STEPS} accum = {effective_batch} effective")
print(f"Warmup steps    : {WARMUP_STEPS}")
print(f"Checkpoint every: {CHECKPOINTING_STEPS} steps")

In [ ]:
import json

output_dir = str(OUTPUT_DIR)
os.makedirs(output_dir, exist_ok=True)

if num_gpus > 1 and needs_sharding:
    # DeepSpeed ZeRO-3: shard model + optimizer across GPUs (required when model > per-GPU VRAM)
    ds_config = {
        "bf16": {"enabled": mixed_precision == "bf16"},
        "fp16": {"enabled": mixed_precision == "fp16"},
        "zero_optimization": {
            "stage": 3,
            "overlap_comm": True,
            "contiguous_gradients": True,
            "reduce_bucket_size": 5e7,
            "stage3_prefetch_bucket_size": 5e7,
            "stage3_param_persistence_threshold": 1e5,
            "stage3_gather_16bit_weights_on_model_save": True,
            "offload_optimizer": {
                "device": "cpu",
                "pin_memory": True,
            },
        },
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "gradient_clipping": MAX_GRAD_NORM,
        "train_batch_size": BATCH_SIZE * num_gpus * GRAD_ACCUM_STEPS,
        "train_micro_batch_size_per_gpu": BATCH_SIZE,
        "steps_per_print": 100,
        "wall_clock_breakdown": False,
    }

    ds_config_path = os.path.join(output_dir, "ds_config.json")
    with open(ds_config_path, "w") as f:
        json.dump(ds_config, f, indent=2)

    accel_config = (
        "compute_environment: LOCAL_MACHINE\n"
        "debug: false\n"
        "deepspeed_config:\n"
        f"  deepspeed_config_file: {ds_config_path}\n"
        "  zero3_init_flag: true\n"
        "distributed_type: DEEPSPEED\n"
        "downcast_bf16: 'no'\n"
        "machine_rank: 0\n"
        "main_training_function: main\n"
        f"mixed_precision: {mixed_precision}\n"
        "num_machines: 1\n"
        f"num_processes: {num_gpus}\n"
        "rdzv_backend: static\n"
        "same_network: true\n"
        "tpu_env: []\n"
        "tpu_use_cluster: false\n"
        "tpu_use_sudo: false\n"
        "use_cpu: false\n"
    )

    accel_config_path = os.path.join(output_dir, "accelerate_config.yaml")
    with open(accel_config_path, "w") as f:
        f.write(accel_config)

    accel_launch_args = f"--config_file {accel_config_path}"
    print(f"🚀 DeepSpeed ZeRO-3 config written:")
    print(f"   {ds_config_path}")
    print(f"   {accel_config_path}")

elif num_gpus > 1:
    # Simple multi-GPU DDP (model fits on each GPU)
    accel_launch_args = f"--multi_gpu --num_processes {num_gpus} --mixed_precision {mixed_precision}"
    print(f"🚀 Multi-GPU DDP on {num_gpus} GPUs")

else:
    # Single GPU
    accel_launch_args = f"--mixed_precision {mixed_precision} --gpu_ids 0"
    print(f"🚀 Single GPU mode")

print(f"\naccelerate launch {accel_launch_args}")

In [ ]:
import time
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_MODE"] = "offline"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["TORCH_NCCL_ENABLE_MONITORING"] = "0"
os.environ["FINETRAINERS_LOG_LEVEL"] = "DEBUG"

fp8_flags = ""
if use_fp8:
    fp8_flags = (
        "    --layerwise_upcasting_modules transformer \\\n"
        "    --layerwise_upcasting_storage_dtype float8_e4m3fn \\\n"
        "    --layerwise_upcasting_skip_modules_pattern "
        'patch_embed pos_embed x_embedder context_embedder "^proj_in$" "^proj_out$" norm \\\n'
    )

script = f"""#!/bin/bash
set -e

export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
export WANDB_MODE=offline
export NCCL_P2P_DISABLE=1
export TORCH_NCCL_ENABLE_MONITORING=0
export FINETRAINERS_LOG_LEVEL=DEBUG

accelerate launch \\
    {accel_launch_args} \\
    {train_script} \\
    --model_name hunyuan_video \\
    --pretrained_model_name_or_path {MODEL_ID} \\
    --data_root "{DATASET_DIR}" \\
    --video_column videos.txt \\
    --caption_column prompts.txt \\
    --id_token {TRIGGER_TOKEN} \\
    --video_resolution_buckets {resolution_buckets} \\
    --caption_dropout_p {CAPTION_DROPOUT} \\
    --dataloader_num_workers 2 \\
    --training_type lora \\
    --seed {SEED} \\
    --batch_size {BATCH_SIZE} \\
    --train_steps {TRAIN_STEPS} \\
    --rank {LORA_RANK} \\
    --lora_alpha {LORA_ALPHA} \\
    --target_modules to_q to_k to_v to_out.0 \\
    --gradient_accumulation_steps {GRAD_ACCUM_STEPS} \\
    --gradient_checkpointing \\
    --max_grad_norm {MAX_GRAD_NORM} \\
    --optimizer adamw \\
    --lr {LEARNING_RATE} \\
    --lr_scheduler {LR_SCHEDULER} \\
    --lr_warmup_steps {WARMUP_STEPS} \\
    --enable_slicing \\
    --enable_tiling \\
    --precompute_conditions \\
    --allow_tf32 \\
    --checkpointing_steps {CHECKPOINTING_STEPS} \\
    --checkpointing_limit 3 \\
    --output_dir "{output_dir}" \\
    --report_to none \\
{fp8_flags}
"""

script_path = os.path.join(output_dir, "run_training.sh")
with open(script_path, "w") as f:
    f.write(script)
os.chmod(script_path, 0o755)

log_path = os.path.join(output_dir, "training_log.txt")

print(f"Training script: {script_path}")
print(f"\n{'='*60}")
print("LAUNCHING TRAINING...")
print(f"{'='*60}\n")

start = time.time()

with open(log_path, "w") as log_file:
    proc = subprocess.Popen(
        ["bash", script_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
        log_file.write(line)
    proc.wait()

elapsed = time.time() - start

if proc.returncode != 0:
    print(f"\n❌ Training failed (exit code {proc.returncode}) after {elapsed/60:.1f} min")
    print(f"Check log: {log_path}")
else:
    print(f"\n✅ Training complete in {elapsed/60:.1f} min")
    print(f"Log: {log_path}")

In [ ]:
output_path = Path(output_dir)

checkpoints = sorted(output_path.glob("checkpoint-*"))
if checkpoints:
    print(f"Checkpoints ({len(checkpoints)}):")
    for cp in checkpoints:
        size_mb = sum(f.stat().st_size for f in cp.rglob("*") if f.is_file()) / 1e6
        print(f"   {cp.name}  ({size_mb:.0f} MB)")
else:
    print("No checkpoints found")

lora_files = list(output_path.rglob("*.safetensors"))
if lora_files:
    print(f"\nLoRA weights:")
    for lf in lora_files:
        size_mb = lf.stat().st_size / 1e6
        print(f"   {lf.relative_to(output_path)}  ({size_mb:.1f} MB)")

if Path(log_path).exists():
    print(f"\nLast 10 lines of training log:")
    with open(log_path) as f:
        lines = f.readlines()
    for line in lines[-10:]:
        print(f"   {line.rstrip()}")

print(f"\nTotal training time: {elapsed/60:.1f} min ({elapsed/3600:.1f} hr)")